# PathSnap — end-to-end walkthrough

A narrated pass through the pipeline on a single trip: raw files, the road
network, how per-observation labels are recovered, candidate generation and
features, the classifier, Viterbi decoding, and the map.

The heavy experiments live in `scripts/` — this notebook is for *seeing* what
each stage does. Run it from the repository root.


## 1. Setup


In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
RAW = ROOT / 'data' / 'raw'
print('repo root:', ROOT)


## 2. What a trip looks like

Every trip ships its own road sub-network, so node and arc ids are *local to
the trip*. Arcs are directed: a two-way street appears twice.


In [ ]:
from pathsnap.trips import load_trip, list_trips

TRIP_ID = '00000001'
trip = load_trip(RAW / 'train', TRIP_ID)

print(f'observations : {trip.n_obs:,}')
print(f'network nodes: {len(trip.nodes):,}')
print(f'network arcs : {len(trip.arcs):,}')
print(f'route arcs   : {len(trip.route):,}')
print()
print('first 3 GPS fixes (lon, lat, t):')
print(trip.track[:3])


In [ ]:
dt = np.diff(trip.timestamps)
print(f'sampling: median {np.median(dt):.1f}s, '
      f'{np.mean(np.isclose(dt, 1.0)) * 100:.1f}% of steps exactly 1s, '
      f'max gap {dt.max():.0f}s')


## 3. The road network

`build_network` projects to local metres and builds the KD-tree used for
candidate retrieval plus the adjacency used for shortest paths.


In [ ]:
from pathsnap.network import build_network

net = build_network(trip)
pd.Series(net.connectivity_summary()).to_frame('value')


Arc lengths are short and heavily skewed — which is why the spatial index
samples points *along* arcs rather than indexing midpoints: a long arc whose
midpoint is far away can still pass right next to the query point.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(net.arc_length, bins=60, range=(0, 300), color='#4C72B0')
ax[0].set(xlabel='arc length (m)', ylabel='arcs', title='Arc length distribution')
ax[1].hist(net.out_degree, bins=range(0, 8), color='#55A868', align='left', rwidth=.8)
ax[1].set(xlabel='out-degree', ylabel='nodes', title='Node out-degree')
plt.tight_layout()


## 4. Recovering per-observation labels

`.route` gives the arcs traversed, not which observation sits on which arc —
here 1,315 observations against 525 route arcs. Picking the nearest route arc
independently lets GPS noise jump backwards and forwards along the route, so
we solve a **monotonic alignment** by dynamic programming instead.


In [ ]:
from pathsnap.labeling import align_observations_to_route

obs_xy = net.projection.to_xy(trip.track[:, 0], trip.track[:, 1])
gt_arc, residual, assign = align_observations_to_route(obs_xy, net, trip.route)

print(f'assignment is monotonic : {bool(np.all(np.diff(assign) >= 0))}')
print(f'residual median         : {np.median(residual):.2f} m')
print(f'residual p95            : {np.percentile(residual, 95):.2f} m')


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(assign, lw=1, color='#4C72B0')
ax[0].set(xlabel='observation', ylabel='index into .route',
          title='Monotonic progression along the reference route')
ax[1].plot(residual, lw=.7, color='#C44E52')
ax[1].set(xlabel='observation', ylabel='distance to assigned arc (m)',
          title='Label residual')
plt.tight_layout()


### The trips where this breaks down

Across the training partition the residuals are sharply bimodal. A few trips
have a `.route` covering only part of the journey, and no alignment can invent
a reference for the rest. Those observations are flagged and excluded rather
than trained on.


In [ ]:
rows = []
for tid in ['00000001', '00000002', '00000033', '00000054', '00000072']:
    t = load_trip(RAW / 'train', tid)
    n = build_network(t)
    xy = n.projection.to_xy(t.track[:, 0], t.track[:, 1])
    _, res, _ = align_observations_to_route(xy, n, t.route)
    track_len = np.hypot(*np.diff(xy, axis=0).T).sum()
    rows.append({'trip': tid, 'observations': t.n_obs,
                 'route_km': n.arc_length[t.route].sum() / 1000,
                 'track_km': track_len / 1000,
                 'coverage': n.arc_length[t.route].sum() / track_len,
                 'median_residual_m': np.median(res),
                 'frac_beyond_30m': (res > 30).mean()})
pd.DataFrame(rows).round(3)


## 5. Candidate generation and features

Arcs within 60 m, the nearest 10 kept, 26 features per (observation, candidate)
pair.


In [ ]:
from pathsnap.dataset import build_trip_frame, candidate_miss_rate
from pathsnap.candidates import FEATURE_NAMES

df = build_trip_frame(trip)
print(f'rows              : {len(df):,}')
print(f'candidates / obs  : {len(df) / trip.n_obs:.2f}')
print(f'candidate miss    : {candidate_miss_rate(df):.4%}')
df.head(8)[['obs_idx', 'arc_id', 'dist_m', 'heading_diff', 'heading_cos', 'label']]


The feature that distinguishes the two directions of a two-way street is
heading, never distance — both directed arcs share identical geometry and so
have identical `dist_m`.


In [ ]:
one = df[df.obs_idx == 0][['arc_id', 'dist_m', 'heading_diff', 'heading_cos', 'label']]
print('candidates for observation 0:')
one.round(3)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, (col, rng) in zip(ax, [('dist_m', (0, 60)), ('heading_diff', (0, np.pi)),
                              ('dist_minus_best', (0, 60))]):
    a.hist(df.loc[df.label == 0, col], bins=40, range=rng, alpha=.65,
           density=True, label='wrong candidate', color='#C44E52')
    a.hist(df.loc[df.label == 1, col], bins=40, range=rng, alpha=.65,
           density=True, label='correct candidate', color='#55A868')
    a.set(xlabel=col, ylabel='density')
ax[0].legend()
plt.suptitle('Feature separation between correct and wrong candidates', fontweight='bold')
plt.tight_layout()


## 6. The trained classifier

Loads whatever `scripts/03` and `scripts/04` produced.


In [ ]:
import joblib

models = sorted((ROOT / 'artifacts' / 'models').glob('*.joblib'))
tuned = [m for m in models if m.name.startswith('tuned_')]
model_path = tuned[0] if tuned else (ROOT / 'artifacts' / 'pathsnap_model.joblib')
model = joblib.load(model_path)
print('using:', model_path.name)


In [ ]:
comparison = pd.read_csv(ROOT / 'results' / 'model_comparison.csv')
comparison[['model', 'algorithm_family', 'cv_macro_f1_mean',
            'test_top1_accuracy', 'test_macro_f1', 'test_log_loss',
            'test_route_continuity']].round(4)


## 7. Classifier only vs classifier + Viterbi

The classifier scores each observation independently, so nothing stops two
consecutive fixes landing on disconnected streets. The decoder uses road-network
travel distance to rule those sequences out.


In [ ]:
from pathsnap.hmm import HMMConfig
from pathsnap.pipeline import PathSnapMatcher, evaluate_against_reference

cfg_path = ROOT / 'results' / 'hmm_config.json'
cfg = HMMConfig(**json.loads(cfg_path.read_text())['config']) if cfg_path.exists() else HMMConfig()
matcher = PathSnapMatcher(model, cfg)

test_trip = load_trip(RAW / 'student_test', list_trips(RAW / 'student_test')[0])
test_net = build_network(test_trip)
result = matcher.match(test_trip, test_net)
ev = evaluate_against_reference(result, test_trip, test_net)

pd.Series({k: v for k, v in ev.items() if isinstance(v, (int, float))}).round(4).to_frame('value')


In [ ]:
print('timings (s):')
for k, v in result.timings.items():
    print(f'  {k:16s} {v:7.2f}')
print()
print('diagnostics:')
for k, v in result.diagnostics.items():
    print(f'  {k:34s} {v}')


## 8. The map

Grey network, blue GPS, green reference, orange classifier-only, red final route.


In [ ]:
from pathsnap.viz import folium_trip_map

m = folium_trip_map(test_trip, test_net,
                    predicted_arcs=result.hmm_arcs,
                    classifier_arcs=result.classifier_arcs,
                    title=f'Trip {test_trip.trip_id}')
m


## 9. Where to go next

- `results/model_comparison.csv` — the full ten-model table
- `results/bayesopt_trials.csv` — every TPE trial
- `results/hmm_comparison_per_trip.csv` — decoder effect per trip
- `results/maps/index.html` — all interactive maps

To serve this to the evaluator UI: `uvicorn api.main:app --host 0.0.0.0 --port 8000`.
